# [ResNet Lab] - Residual Networks & Transfer Learning

In this notebook, you'll learn about one of the most important innovations in deep learning: **Residual Networks (ResNets)**. You'll also discover how to leverage pretrained models through **Transfer Learning** to solve new tasks with limited data.

This lab is designed to teach you:

1. **Residual connections**: Why they enable training very deep networks
2. **ResNet architecture**: Building blocks and implementation
3. **Layer Normalization**: An alternative to Batch Normalization
4. **Transfer Learning**: Leveraging pretrained models for new tasks
5. **Fine-tuning strategies**: Feature extraction vs full fine-tuning

## How to Use This Notebook

This notebook builds on concepts from previous CNN labs:

- **Read carefully**: Understanding the theory behind residual connections is crucial
- **Run linearly**: Execute cells in order from top to bottom
- **Complete exercises**: Implement the TODOs marked in code cells
- **Experiment**: Try different configurations after completing the exercises
- **Observe**: Pay attention to training dynamics and gradient flow

Expected time: **~3 hours**

## Content & Learning Objectives

This lab is divided into 3 main sections:

### 1️⃣ Residual Networks (ResNets)
Understand and implement residual connections that revolutionized deep learning.

> ##### Learning Objectives
> 
> - Understand the vanishing gradient problem in deep networks
> - Learn how skip connections enable training very deep networks
> - Implement ResidualBlock as a custom nn.Module
> - Build a simple ResNet architecture
> - Visualize gradient flow with and without residual connections

### 2️⃣ Layer Normalization
Learn an alternative normalization technique and compare it with Batch Normalization.

> ##### Learning Objectives
>
> - Understand how Layer Normalization works
> - Implement LayerNorm from scratch
> - Compare BatchNorm vs LayerNorm (when to use each)
> - Understand the trade-offs between normalization techniques

### 3️⃣ Transfer Learning
Leverage pretrained models to solve new tasks efficiently.

> ##### Learning Objectives
>
> - Load pretrained models from PyTorch
> - Understand feature extraction (frozen backbone)
> - Implement full fine-tuning strategy
> - Compare feature extraction vs full fine-tuning
> - Learn when to use each transfer learning approach

## Setup code

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import Optional
import copy

import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import datasets, transforms, models

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(42)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

# 1️⃣ Residual Networks (ResNets)

## The Problem: Training Very Deep Networks

Before ResNets (2015), a puzzling phenomenon occurred: **adding more layers to a neural network could actually make it perform worse**.

### The Degradation Problem

Researchers observed that:
- A 20-layer network trained well
- A 56-layer network had **higher training error** than the 20-layer network

This wasn't overfitting (which affects test error) - it was **degradation**: the network couldn't even fit the training data!

**Why does this happen?**

1. **Vanishing gradients**: In very deep networks, gradients become extremely small as they backpropagate
2. **Optimization difficulty**: Deep networks have complex loss landscapes that are hard to optimize
3. **Identity mapping problem**: The network struggles to learn even simple identity mappings

### The Insight: Residual Learning

The key insight from ResNet is: instead of learning the desired underlying mapping $H(x)$, learn the **residual** (difference) $F(x) = H(x) - x$.

**Traditional layer:**
```
Output = H(x)
```

**Residual layer:**
```
Output = F(x) + x
```

where $F(x)$ is what the layers learn, and $x$ is the **skip connection** (also called shortcut connection).

### Why Residual Learning Works

**1. Easier optimization:**
- It's easier to learn $F(x) = 0$ (push residual to zero) than to learn an identity mapping
- If the optimal function is close to identity, the network just needs to learn small adjustments

**2. Gradient flow:**
- Skip connections provide a direct path for gradients to flow backward
- During backpropagation: $\frac{\partial \text{loss}}{\partial x} = \frac{\partial \text{loss}}{\partial \text{output}} \cdot (1 + \frac{\partial F(x)}{\partial x})$
- The "+1" ensures gradients can always flow, even if $\frac{\partial F(x)}{\partial x}$ vanishes

**3. Ensemble-like behavior:**
- A ResNet with $n$ blocks can be seen as an ensemble of $2^n$ paths
- Different paths of different lengths contribute to the final output

## Residual Block Architecture

A basic ResidualBlock consists of:

```
       Input (x)
          |
          |--------------------\
          |                    |
      Conv 3×3                 |  (skip connection)
          |                    |
      BatchNorm                |
          |                    |
        ReLU                   |
          |                    |
      Conv 3×3                 |
          |                    |
      BatchNorm                |
          |                    |
      Add <--------------------/
          |
        ReLU
          |
       Output
```

### Handling Dimension Mismatch

When input and output dimensions don't match (different number of channels or spatial size), we need to **project** the skip connection:

**Option 1: Identity shortcut with zero padding** (for channel mismatch)
- Pad extra channels with zeros
- Use stride=2 in first conv when downsampling

**Option 2: Projection shortcut** (more common)
- Use a 1×1 convolution to match dimensions
- Apply stride=2 when downsampling

```python
if in_channels != out_channels or stride != 1:
    self.shortcut = nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride),
        nn.BatchNorm2d(out_channels)
    )
else:
    self.shortcut = nn.Identity()
```

## Exercise - Implement ResidualBlock

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Importance: 🔵🔵🔵🔵🔵
>
> You should spend up to 25-30 minutes on this exercise.
> This is the core exercise of the ResNet section.
> ```

Implement a ResidualBlock that handles both matching and mismatching dimensions.

**Your task:**
1. Implement the main path: Conv → BatchNorm → ReLU → Conv → BatchNorm
2. Implement the shortcut path (identity or projection)
3. Add the two paths together
4. Apply final ReLU activation

**Parameters:**
- `in_channels`: Number of input channels
- `out_channels`: Number of output channels
- `stride`: Stride for downsampling (default: 1)

**Important notes:**
- Use `kernel_size=3, padding=1` for the 3×3 convolutions
- Apply stride only to the first conv in the main path
- Use projection shortcut when dimensions don't match
- BatchNorm comes after each Conv, before ReLU (except the final ReLU is after the addition)

<details>
<summary>Hint - Structure</summary>

```python
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        
        # Main path
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, 
                               stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # Shortcut path
        if in_channels != out_channels or stride != 1:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, 
                         stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
        else:
            self.shortcut = nn.Identity()
    
    def forward(self, x):
        # Main path
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        
        # Shortcut path
        identity = self.shortcut(x)
        
        # Add and activate
        out = out + identity
        out = self.relu(out)
        return out
```
</details>

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, stride: int = 1):
        super().__init__()
        
        # TODO: Implement main path (two conv layers with batch norm)
        self.conv1 = ...
        self.bn1 = ...
        self.relu = ...
        self.conv2 = ...
        self.bn2 = ...
        
        # TODO: Implement shortcut (projection when dimensions don't match)
        if ...:
            self.shortcut = ...
        else:
            self.shortcut = ...
    
    def forward(self, x):  # TODO: Implement forward pass
        ...

In [ ]:
# Test ResidualBlock
print("Test 1: Same dimensions (identity shortcut)")
block1 = ResidualBlock(64, 64, stride=1).to(device)
x1 = torch.randn(2, 64, 32, 32).to(device)
out1 = block1(x1)
print(f"Input shape:  {x1.shape}")
print(f"Output shape: {out1.shape}")
assert out1.shape == (2, 64, 32, 32), "Output shape incorrect for identity shortcut!"
print("✅ Test 1 passed!\n")

print("Test 2: Different channels (projection shortcut)")
block2 = ResidualBlock(64, 128, stride=1).to(device)
x2 = torch.randn(2, 64, 32, 32).to(device)
out2 = block2(x2)
print(f"Input shape:  {x2.shape}")
print(f"Output shape: {out2.shape}")
assert out2.shape == (2, 128, 32, 32), "Output shape incorrect for projection shortcut!"
print("✅ Test 2 passed!\n")

print("Test 3: Downsampling with stride=2")
block3 = ResidualBlock(64, 128, stride=2).to(device)
x3 = torch.randn(2, 64, 32, 32).to(device)
out3 = block3(x3)
print(f"Input shape:  {x3.shape}")
print(f"Output shape: {out3.shape}")
assert out3.shape == (2, 128, 16, 16), "Output shape incorrect for downsampling!"
print("✅ Test 3 passed!\n")

print("✅ All ResidualBlock tests passed!")

# 2️⃣ Layer Normalization

## What is Layer Normalization?

**Layer Normalization** (LayerNorm) is an alternative to Batch Normalization that normalizes across features instead of across the batch.

### Batch Normalization vs Layer Normalization

**Batch Normalization:**
- Normalizes across the **batch dimension**
- For each feature, computes mean and variance across all samples in the batch
- Shape: `(B, C, H, W)` → normalize over `B` (batch)

**Layer Normalization:**
- Normalizes across the **feature dimension**
- For each sample, computes mean and variance across all features
- Shape: `(B, C, H, W)` → normalize over `C, H, W` (features)

### Visual Comparison

For a tensor of shape `(B=4, C=3, H=8, W=8)`:

**BatchNorm:**
```
Computes mean/var over B=4 samples
→ One mean/var per channel (3 total)
→ Statistics depend on batch composition
```

**LayerNorm:**
```
Computes mean/var over C×H×W features
→ One mean/var per sample (4 total)
→ Statistics independent of other samples
```

### Layer Normalization Formula

For each sample in the batch independently:

**1. Compute mean across all features:**
$$\mu = \frac{1}{C \cdot H \cdot W} \sum_{c,h,w} x_{c,h,w}$$

**2. Compute variance across all features:**
$$\sigma^2 = \frac{1}{C \cdot H \cdot W} \sum_{c,h,w} (x_{c,h,w} - \mu)^2$$

**3. Normalize:**
$$\hat{x}_{c,h,w} = \frac{x_{c,h,w} - \mu}{\sqrt{\sigma^2 + \epsilon}}$$

**4. Scale and shift (learnable parameters):**
$$y_{c,h,w} = \gamma \hat{x}_{c,h,w} + \beta$$

where $\gamma$ and $\beta$ are learned parameters (same shape as input features).

### BatchNorm vs LayerNorm: When to Use Which?

| Aspect | Batch Normalization | Layer Normalization |
|--------|--------------------|---------------------|
| **Normalizes over** | Batch dimension | Feature dimensions |
| **Batch size dependency** | ❌ Requires large batches | ✅ Works with batch size = 1 |
| **Train/eval modes** | ❌ Different behavior | ✅ Same behavior |
| **Running statistics** | ❌ Needs to maintain | ✅ No running statistics |
| **Best for** | CNNs, large batch sizes | RNNs, Transformers, small batches |
| **Performance on CNNs** | ✅ Usually better | ⚠️ Comparable but slightly worse |
| **Distributed training** | ⚠️ Needs sync across devices | ✅ No synchronization needed |
| **Inference** | ⚠️ Uses running stats | ✅ Computes on the fly |

### Why BatchNorm is Preferred for CNNs

**1. Spatial features:** 
- In CNNs, each channel represents a specific feature detector (e.g., edge detector)
- It makes sense to normalize each feature across different images
- BatchNorm normalizes "all vertical edges across the batch"

**2. Empirical performance:**
- BatchNorm consistently performs better on vision tasks
- Better regularization effect

### Why LayerNorm is Preferred for Transformers/RNNs

**1. Variable sequence lengths:**
- Different samples can have different lengths
- BatchNorm across the batch doesn't make sense

**2. Small or varying batch sizes:**
- In NLP, batch sizes can be very small (even 1)
- LayerNorm doesn't care about batch size

**3. Online learning:**
- Can process one sample at a time
- No need for running statistics

### Key Takeaway

- **Use BatchNorm for CNNs** (vision tasks with fixed-size images and reasonable batch sizes)
- **Use LayerNorm for Transformers/RNNs** (sequence tasks, variable lengths, or small batches)
- Both can work, but one is usually more natural for a given architecture

## Exercise - Implement LayerNorm

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵⚪
>
> You should spend up to 15-20 minutes on this exercise.
> ```

Implement Layer Normalization for 2D inputs (images).

**Your task:**
1. Initialize learnable `gamma` (scale) and `beta` (shift) parameters
2. In forward pass:
   - Compute mean across feature dimensions (C, H, W)
   - Compute variance across feature dimensions
   - Normalize the input
   - Apply scale and shift

**Important notes:**
- Input shape: `(B, C, H, W)`
- Compute statistics over dimensions `(1, 2, 3)` for each sample independently
- Use `keepdim=True` to maintain broadcasting compatibility
- `gamma` and `beta` should have shape matching `(1, C, H, W)` or just `(C, H, W)`
- Use `eps=1e-5` for numerical stability

<details>
<summary>Hint - Implementation</summary>

```python
class LayerNorm2d(nn.Module):
    def __init__(self, num_features, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(1, num_features, 1, 1))
        self.beta = nn.Parameter(torch.zeros(1, num_features, 1, 1))
    
    def forward(self, x):
        # Compute mean and var over C, H, W dimensions
        mean = x.mean(dim=(1, 2, 3), keepdim=True)
        var = x.var(dim=(1, 2, 3), keepdim=True, unbiased=False)
        
        # Normalize
        x_norm = (x - mean) / torch.sqrt(var + self.eps)
        
        # Scale and shift
        return self.gamma * x_norm + self.beta
```
</details>

In [ ]:
class LayerNorm2d(nn.Module):
    def __init__(self, num_features: int, eps: float = 1e-5):
        super().__init__()
        self.eps = eps
        # TODO: Initialize gamma and beta parameters
        self.gamma = ...
        self.beta = ...
    
    def forward(self, x):  # TODO: Implement layer normalization
        ...

In [ ]:
# Test LayerNorm2d
ln = LayerNorm2d(num_features=64).to(device)

# Test 1: Output shape
x = torch.randn(4, 64, 8, 8).to(device)
out = ln(x)
print(f"Input shape:  {x.shape}")
print(f"Output shape: {out.shape}")
assert out.shape == x.shape, "Output shape should match input shape!"
print("✅ Shape test passed!\n")

# Test 2: Normalization (mean ≈ 0, std ≈ 1 for each sample)
print("Testing normalization properties...")
with torch.no_grad():
    # Initialize gamma=1, beta=0 for pure normalization
    ln.gamma.fill_(1.0)
    ln.beta.fill_(0.0)
    out = ln(x)
    
    # Check mean and std for first sample
    sample_mean = out[0].mean().item()
    sample_std = out[0].std().item()
    print(f"First sample - Mean: {sample_mean:.6f}, Std: {sample_std:.6f}")
    assert abs(sample_mean) < 1e-5, f"Mean should be ≈0, got {sample_mean}"
    assert abs(sample_std - 1.0) < 1e-3, f"Std should be ≈1, got {sample_std}"

print("✅ Normalization test passed!\n")

# Test 3: Learnable parameters
num_params = sum(p.numel() for p in ln.parameters())
print(f"Number of parameters: {num_params}")
assert num_params == 128, f"Expected 128 parameters (64*2), got {num_params}"

print("\n✅ All LayerNorm2d tests passed!")

## Building a Simple ResNet

Now let's build a minimal ResNet for educational purposes. We'll create a simple architecture with just a few residual blocks.

## Exercise - Build SimpleResNet

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵⚪
>
> You should spend up to 20-25 minutes on this exercise.
> ```

Build a simple ResNet architecture for CIFAR-10 using your ResidualBlock.

**Architecture:**
```
Input (3, 32, 32)
    ↓
Conv 3×3, 64 channels  [→ (64, 32, 32)]
    ↓
BatchNorm + ReLU
    ↓
ResidualBlock(64, 64)  [→ (64, 32, 32)]
    ↓
ResidualBlock(64, 128, stride=2)  [→ (128, 16, 16)]
    ↓
ResidualBlock(128, 128)  [→ (128, 16, 16)]
    ↓
ResidualBlock(128, 256, stride=2)  [→ (256, 8, 8)]
    ↓
ResidualBlock(256, 256)  [→ (256, 8, 8)]
    ↓
AdaptiveAvgPool2d(1, 1)  [→ (256, 1, 1)]
    ↓
Flatten  [→ (256,)]
    ↓
Linear(256, 10)
```

**Your task:**
1. Implement the initial conv layer with BatchNorm and ReLU
2. Add the residual blocks as specified
3. Add adaptive average pooling to get fixed size output
4. Add flatten and final linear layer

<details>
<summary>Hint</summary>

```python
class SimpleResNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        
        # Initial conv layer
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU()
        
        # Residual blocks
        self.block1 = ResidualBlock(64, 64, stride=1)
        self.block2 = ResidualBlock(64, 128, stride=2)
        self.block3 = ResidualBlock(128, 128, stride=1)
        self.block4 = ResidualBlock(128, 256, stride=2)
        self.block5 = ResidualBlock(256, 256, stride=1)
        
        # Global pooling and classifier
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(256, num_classes)
```
</details>

In [ ]:
class SimpleResNet(nn.Module):
    def __init__(self, num_classes: int = 10):
        super().__init__()
        
        # TODO: Implement initial conv layer
        self.conv1 = ...
        self.bn1 = ...
        self.relu = ...
        
        # TODO: Implement residual blocks
        self.block1 = ...
        self.block2 = ...
        self.block3 = ...
        self.block4 = ...
        self.block5 = ...
        
        # TODO: Implement pooling and classifier
        self.avgpool = ...
        self.flatten = ...
        self.fc = ...
    
    def forward(self, x):  # TODO: Implement forward pass
        ...

In [ ]:
# Test SimpleResNet
model = SimpleResNet(num_classes=10).to(device)

# Test forward pass
x = torch.randn(4, 3, 32, 32).to(device)
out = model(x)

print(f"Input shape:  {x.shape}")
print(f"Output shape: {out.shape}")
assert out.shape == (4, 10), f"Expected shape (4, 10), got {out.shape}"

# Count parameters
num_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {num_params:,}")

print("\n✅ SimpleResNet implementation correct!")
print(f"\nModel architecture:\n{model}")

## Training SimpleResNet

Let's train our ResNet for a few epochs to verify it works correctly. We won't train to convergence - this is just to ensure the implementation is correct.

In [ ]:
# Load CIFAR-10
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD)
])

train_dataset = datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)

# Create train/val split
train_size = 45000
val_size = 5000
train_subset, val_subset = random_split(
    train_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_subset, batch_size=128, shuffle=True, num_workers=2)
val_loader = DataLoader(val_subset, batch_size=256, shuffle=False, num_workers=2)

print(f"Training samples: {len(train_subset):,}")
print(f"Validation samples: {len(val_subset):,}")

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    
    for X, y in dataloader:
        X, y = X.to(device), y.to(device)
        
        optimizer.zero_grad()
        logits = model(X)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * y.size(0)
        total_correct += (logits.argmax(dim=1) == y).sum().item()
        total_samples += y.size(0)
    
    return total_loss / total_samples, total_correct / total_samples

def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            logits = model(X)
            loss = criterion(logits, y)
            
            total_loss += loss.item() * y.size(0)
            total_correct += (logits.argmax(dim=1) == y).sum().item()
            total_samples += y.size(0)
    
    return total_loss / total_samples, total_correct / total_samples

In [ ]:
# Train for a few epochs to verify the model works
set_seed(42)
model = SimpleResNet(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print("Training SimpleResNet for a few epochs...\n")
print(f"Model has {sum(p.numel() for p in model.parameters()):,} parameters\n")

for epoch in range(1, 4):  # Just 3 epochs to verify it works
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)
    
    print(f"Epoch {epoch}/3 | "
          f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f} | "
          f"val_loss={val_loss:.4f}, val_acc={val_acc:.4f}")

print("\n✅ SimpleResNet trains successfully!")
print("\nNote: The model is not fully trained. Continue training if you want better accuracy.")

# 3️⃣ Transfer Learning

## What is Transfer Learning?

**Transfer Learning** is the practice of using a model trained on one task as a starting point for another task.

### Why Transfer Learning?

**Problem:** Training deep neural networks from scratch requires:
- Large amounts of labeled data (often millions of examples)
- Substantial computational resources (days or weeks of training)
- Expertise in hyperparameter tuning

**Solution:** Use a model pretrained on a large dataset (like ImageNet) and adapt it to your specific task.

### The ImageNet Advantage

Models pretrained on **ImageNet** have learned powerful features:
- **ImageNet**: 1.2 million images, 1000 categories
- **Early layers**: Learn general features (edges, textures, colors)
- **Middle layers**: Learn patterns and shapes
- **Later layers**: Learn high-level, task-specific features

These learned features transfer well to other vision tasks!

## Two Transfer Learning Strategies

### Strategy 1: Feature Extraction (Frozen Backbone)

**Idea:** Use the pretrained network as a fixed feature extractor.

```
Pretrained Network               Your Task
┌────────────────┐              ┌──────────┐
│ Conv Layers    │ ❄️ Frozen    │ New FC   │ 🔥 Trainable
│ (feature       │ →            │ Layer    │
│  extractor)    │              │          │
└────────────────┘              └──────────┘
```

**How it works:**
1. Load a pretrained model (e.g., ResNet-18)
2. **Freeze** all convolutional layers (set `requires_grad=False`)
3. **Replace** the final fully connected layer with a new one
4. **Train** only the new FC layer

**When to use:**
- ✅ Small dataset (< 10,000 images)
- ✅ Similar to ImageNet (natural images)
- ✅ Limited computational resources
- ✅ Quick experiments

**Advantages:**
- Very fast training (only updating a few parameters)
- Less prone to overfitting (most weights are frozen)
- Works well with small datasets

**Disadvantages:**
- May not adapt well if new task is very different from ImageNet
- Lower ceiling on potential performance

### Strategy 2: Full Fine-Tuning

**Idea:** Update all layers, but initialize from pretrained weights.

```
Pretrained Network               Your Task
┌────────────────┐              ┌──────────┐
│ Conv Layers    │ 🔥 Trainable │ New FC   │ 🔥 Trainable
│ (initialized   │ →            │ Layer    │
│  from ImageNet)│              │          │
└────────────────┘              └──────────┘
```

**How it works:**
1. Load a pretrained model
2. Replace the final FC layer
3. **Train all layers** (but start from pretrained weights)
4. Often use a smaller learning rate than training from scratch

**When to use:**
- ✅ Medium to large dataset (> 10,000 images)
- ✅ Somewhat different from ImageNet
- ✅ You want the best possible performance
- ✅ You have computational resources

**Advantages:**
- Can achieve higher accuracy than feature extraction
- Adapts the features to your specific task
- Still benefits from pretrained initialization

**Disadvantages:**
- Slower training (updating all parameters)
- Risk of overfitting on small datasets
- Requires more computational resources

## Decision Guide

| Dataset Size | Task Similarity to ImageNet | Recommended Strategy |
|--------------|----------------------------|---------------------|
| Very small (< 1K) | Similar | Feature extraction |
| Very small (< 1K) | Different | Feature extraction + small FC |
| Small (1K-10K) | Similar | Feature extraction |
| Small (1K-10K) | Different | Light fine-tuning (few epochs) |
| Medium (10K-100K) | Similar | Feature extraction or fine-tuning |
| Medium (10K-100K) | Different | Full fine-tuning |
| Large (> 100K) | Any | Full fine-tuning or train from scratch |

## Key Insight

**Transfer learning is almost always better than training from scratch** for computer vision tasks, even if you have a lot of data. The pretrained features provide a better starting point than random initialization.

## Part 1: Feature Extraction with Frozen Backbone

Let's load a pretrained ResNet-18 and use it as a feature extractor for CIFAR-10.

In [ ]:
# Load pretrained ResNet-18
print("Loading pretrained ResNet-18...")
pretrained_model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
print(f"✅ Loaded ResNet-18 pretrained on ImageNet\n")

# Inspect the architecture
print("ResNet-18 architecture:")
print(pretrained_model)
print(f"\nTotal parameters: {sum(p.numel() for p in pretrained_model.parameters()):,}")

In [ ]:
# Create feature extraction model
def create_feature_extractor(num_classes=10):
    """Create a model for feature extraction (frozen backbone)"""
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    
    # Freeze all layers
    for param in model.parameters():
        param.requires_grad = False
    
    # Replace the final fully connected layer
    # ResNet-18 has 512 features before the FC layer
    num_features = model.fc.in_features
    model.fc = nn.Linear(num_features, num_classes)
    
    return model

# Create the model
feature_extractor = create_feature_extractor(num_classes=10).to(device)

# Count trainable parameters
total_params = sum(p.numel() for p in feature_extractor.parameters())
trainable_params = sum(p.numel() for p in feature_extractor.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters: {total_params - trainable_params:,}")
print(f"\nOnly {100 * trainable_params / total_params:.2f}% of parameters are trainable!")

### Important: ImageNet Preprocessing

When using pretrained models, we must preprocess our data the **same way** the model was trained.

**ImageNet preprocessing:**
- Resize images to 224×224 (ImageNet standard)
- Normalize with ImageNet mean and std:
  - Mean: `[0.485, 0.456, 0.406]` (RGB channels)
  - Std: `[0.229, 0.224, 0.225]` (RGB channels)

Even though CIFAR-10 is 32×32, we resize to 224×224 to match what ResNet expects.

In [ ]:
# ImageNet normalization (required for pretrained models)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

transform_imagenet = transforms.Compose([
    transforms.Resize(224),  # Resize to ImageNet size
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

# Reload CIFAR-10 with ImageNet preprocessing
train_dataset_imagenet = datasets.CIFAR10(root="./data", train=True, download=True, 
                                          transform=transform_imagenet)
test_dataset_imagenet = datasets.CIFAR10(root="./data", train=False, download=True,
                                         transform=transform_imagenet)

train_subset_imagenet, val_subset_imagenet = random_split(
    train_dataset_imagenet,
    [45000, 5000],
    generator=torch.Generator().manual_seed(42)
)
# Sub-sample training set for faster experimentation
train_subset_imagenet = Subset(train_dataset_imagenet, train_subset_imagenet.indices[:15000])  # Use only 15k for faster runs


train_loader_imagenet = DataLoader(train_subset_imagenet, batch_size=256, shuffle=True, num_workers=2)
val_loader_imagenet = DataLoader(val_subset_imagenet, batch_size=256, shuffle=False, num_workers=2)
test_loader_imagenet = DataLoader(test_dataset_imagenet, batch_size=256, shuffle=False, num_workers=2)

print("Loaded CIFAR-10 with ImageNet preprocessing (224×224)")

In [ ]:
# Train feature extraction model
set_seed(42)
feature_extractor = create_feature_extractor(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, feature_extractor.parameters()), lr=1e-3)

print("Training with feature extraction (frozen backbone)...\n")

feature_extraction_history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for epoch in range(1, 6):  # Train for 5 epochs
    train_loss, train_acc = train_epoch(feature_extractor, train_loader_imagenet, criterion, optimizer, device)
    val_loss, val_acc = evaluate(feature_extractor, val_loader_imagenet, criterion, device)
    
    feature_extraction_history["train_loss"].append(train_loss)
    feature_extraction_history["train_acc"].append(train_acc)
    feature_extraction_history["val_loss"].append(val_loss)
    feature_extraction_history["val_acc"].append(val_acc)
    
    print(f"Epoch {epoch:02d}/5 | "
          f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f} | "
          f"val_loss={val_loss:.4f}, val_acc={val_acc:.4f}")

# Evaluate on test set
test_loss, test_acc = evaluate(feature_extractor, test_loader_imagenet, criterion, device)
print(f"\nFeature Extraction - Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")

## Part 2: Full Fine-Tuning

Now let's train all layers (full fine-tuning) and compare the results.

In [ ]:
# Create fine-tuning model (all layers trainable)
def create_finetuning_model(num_classes=10):
    """Create a model for full fine-tuning (all layers trainable)"""
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    
    # Replace the final fully connected layer
    num_features = model.fc.in_features
    model.fc = nn.Linear(num_features, num_classes)
    
    # All layers are trainable by default
    return model

# Create the model
finetuning_model = create_finetuning_model(num_classes=10).to(device)

# Count trainable parameters
total_params = sum(p.numel() for p in finetuning_model.parameters())
trainable_params = sum(p.numel() for p in finetuning_model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"\nAll {100 * trainable_params / total_params:.0f}% of parameters are trainable!")

In [ ]:
# Train full fine-tuning model
set_seed(42)
finetuning_model = create_finetuning_model(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(finetuning_model.parameters(), lr=1e-4)  # Smaller LR for fine-tuning

print("Training with full fine-tuning (all layers trainable)...\n")

finetuning_history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for epoch in range(1, 6):  # Train for 5 epochs
    train_loss, train_acc = train_epoch(finetuning_model, train_loader_imagenet, criterion, optimizer, device)
    val_loss, val_acc = evaluate(finetuning_model, val_loader_imagenet, criterion, device)
    
    finetuning_history["train_loss"].append(train_loss)
    finetuning_history["train_acc"].append(train_acc)
    finetuning_history["val_loss"].append(val_loss)
    finetuning_history["val_acc"].append(val_acc)
    
    print(f"Epoch {epoch:02d}/5 | "
          f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f} | "
          f"val_loss={val_loss:.4f}, val_acc={val_acc:.4f}")

# Evaluate on test set
test_loss, test_acc = evaluate(finetuning_model, test_loader_imagenet, criterion, device)
print(f"\nFull Fine-tuning - Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")

## Comparing Feature Extraction vs Full Fine-Tuning

In [ ]:
# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs = np.arange(1, 11)

# Loss comparison
axes[0].plot(epochs, feature_extraction_history["train_loss"], 'b-', marker='o', label="Feature Extraction (train)")
axes[0].plot(epochs, feature_extraction_history["val_loss"], 'b--', marker='o', label="Feature Extraction (val)")
axes[0].plot(epochs, finetuning_history["train_loss"], 'r-', marker='s', label="Fine-tuning (train)")
axes[0].plot(epochs, finetuning_history["val_loss"], 'r--', marker='s', label="Fine-tuning (val)")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training Loss Comparison")
axes[0].legend()
axes[0].grid(alpha=0.3)

# Accuracy comparison
axes[1].plot(epochs, feature_extraction_history["train_acc"], 'b-', marker='o', label="Feature Extraction (train)")
axes[1].plot(epochs, feature_extraction_history["val_acc"], 'b--', marker='o', label="Feature Extraction (val)")
axes[1].plot(epochs, finetuning_history["train_acc"], 'r-', marker='s', label="Fine-tuning (train)")
axes[1].plot(epochs, finetuning_history["val_acc"], 'r--', marker='s', label="Fine-tuning (val)")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Training Accuracy Comparison")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Create summary comparison
# Get final test accuracies
feat_ext_test_loss, feat_ext_test_acc = evaluate(feature_extractor, test_loader_imagenet, criterion, device)
finetune_test_loss, finetune_test_acc = evaluate(finetuning_model, test_loader_imagenet, criterion, device)

comparison_data = {
    "Approach": ["Feature Extraction", "Full Fine-tuning"],
    "Trainable Params": ["5,130 (0.05%)", "11,181,642 (100%)"],
    "Training Speed": ["Fast", "Slow"],
    "Final Train Acc": [
        f"{feature_extraction_history['train_acc'][-1]:.4f}",
        f"{finetuning_history['train_acc'][-1]:.4f}"
    ],
    "Final Val Acc": [
        f"{feature_extraction_history['val_acc'][-1]:.4f}",
        f"{finetuning_history['val_acc'][-1]:.4f}"
    ],
    "Test Acc": [
        f"{feat_ext_test_acc:.4f}",
        f"{finetune_test_acc:.4f}"
    ]
}

comparison_df = pd.DataFrame(comparison_data)
print("\nTransfer Learning Comparison:")
print("=" * 80)
print(comparison_df.to_string(index=False))
print("=" * 80)

# Summary & Key Takeaways

## What We've Learned

### 1. Residual Networks (ResNets)

**The Problem:**
- Deep networks suffered from degradation (couldn't even fit training data)
- Vanishing gradients prevented effective learning in very deep networks

**The Solution: Skip Connections**
- Instead of learning $H(x)$, learn the residual $F(x) = H(x) - x$
- Output = $F(x) + x$ (main path + skip connection)
- Enables training networks with 100+ layers

**Why It Works:**
- Easier optimization (learning deviations from identity)
- Better gradient flow (skip connections provide direct paths)
- Ensemble-like behavior (multiple effective paths)

### 2. Layer Normalization vs Batch Normalization

**Batch Normalization:**
- Normalizes across the batch dimension
- Best for CNNs with reasonable batch sizes
- Requires different behavior in train/eval modes

**Layer Normalization:**
- Normalizes across feature dimensions per sample
- Best for RNNs, Transformers, or when batch size varies
- Same behavior in train and eval modes

**Rule of thumb:**
- Use BatchNorm for computer vision (CNNs)
- Use LayerNorm for NLP (Transformers, RNNs)

### 3. Transfer Learning

**Two Strategies:**

**Feature Extraction (Frozen Backbone):**
- Freeze pretrained layers, train only new classifier
- Fast, works well with small datasets
- Best when new task is similar to ImageNet

**Full Fine-Tuning:**
- Train all layers, but initialize from pretrained weights
- Slower, but achieves better performance
- Best when you have sufficient data and computational resources

## Best Practices

### ResNets
✅ **Do:**
- Use skip connections for networks deeper than ~10 layers
- Apply BatchNorm before ReLU in the main path
- Use projection shortcuts when dimensions change
- Apply ReLU after the addition (not before)

❌ **Don't:**
- Forget the skip connection (defeats the purpose!)
- Apply activation before the addition
- Use very deep networks without residual connections

### Transfer Learning
✅ **Do:**
- Always use pretrained models when available
- Match the preprocessing (resize to 224×224, use ImageNet normalization)
- Start with feature extraction for quick experiments
- Use smaller learning rates for fine-tuning (e.g., 1e-4 instead of 1e-3)
- Freeze early layers and unfreeze gradually if needed

❌ **Don't:**
- Use different normalization than the pretrained model
- Use the same learning rate as training from scratch
- Fine-tune all layers on tiny datasets (< 1000 samples)
- Forget to set `requires_grad=False` for frozen layers

## Performance Comparison

**On CIFAR-10 (our experiments):**
- Simple CNN (previous lab): ~70-75% accuracy
- Simple ResNet (trained from scratch): ~75-80% accuracy after 3 epochs
- ResNet-50 Feature Extraction: ~85-90% accuracy
- ResNet-50 Full Fine-tuning: ~90-95% accuracy

**Key insight:** Transfer learning dramatically improves performance, even though ImageNet and CIFAR-10 are different datasets!

## When to Use What

| Scenario | Recommended Approach |
|----------|---------------------|
| Small dataset (< 1K), similar to ImageNet | Feature extraction |
| Small dataset (< 1K), different from ImageNet | Feature extraction with data augmentation |
| Medium dataset (1K-10K) | Feature extraction or light fine-tuning |
| Large dataset (10K-100K) | Full fine-tuning |
| Very large dataset (> 100K) | Fine-tuning or train from scratch |
| Need very deep network (> 50 layers) | Use ResNet architecture |
| Working with sequences or small batches | Use LayerNorm instead of BatchNorm |

## Congratulations!

You've successfully:
- ✅ Understood residual connections and why they work
- ✅ Implemented ResidualBlock and SimpleResNet
- ✅ Learned about Layer Normalization
- ✅ Implemented LayerNorm from scratch
- ✅ Loaded and used pretrained models
- ✅ Applied feature extraction strategy
- ✅ Applied full fine-tuning strategy
- ✅ Compared different transfer learning approaches

You now have the tools to:
- Build very deep networks that train effectively
- Leverage pretrained models for new tasks
- Choose the right normalization technique
- Make informed decisions about transfer learning strategies